### Playing around with the requests library

In [123]:
import requests

routeURL = "https://pt.mytransitride.com/api/Route"
vehicleStatusesURL = "https://pt.mytransitride.com/api/VehicleStatuses"

route = requests.get("https://pt.mytransitride.com/api/Route")

route.json()

[{'routeID': 2627,
  'routeNumber': 1429,
  'routeCode': '8',
  'routeName': 'Monaghan',
  'color': 'FF33CC',
  'headSign': '8 - Fleming College',
  'patternID': 11215,
  'patternCode': '8-South-36 (8)',
  'patternName': '8-South-36 (8)',
  'patternNumber': 8935,
  'directionID': 2,
  'directionName': 'South',
  'serviceID': None,
  'serviceName': None,
  'arrivalTimes': None},
 {'routeID': 2628,
  'routeNumber': 1430,
  'routeCode': '9',
  'routeName': 'Parkhill',
  'color': 'A92023',
  'headSign': '9 - Fleming College',
  'patternID': 11223,
  'patternCode': '9-West-79 (9)',
  'patternName': '9-West-79 (9)',
  'patternNumber': 10031,
  'directionID': 4,
  'directionName': 'West',
  'serviceID': None,
  'serviceName': None,
  'arrivalTimes': None},
 {'routeID': 2632,
  'routeNumber': 1434,
  'routeCode': '23',
  'routeName': 'CB Red',
  'color': 'FC0008',
  'headSign': '23 - Red Community Bus',
  'patternID': 11251,
  'patternCode': '23-Inbound-31 (23)',
  'patternName': '23-Inbound-3

In [124]:
with open("patterns.txt") as patterns:
    patternIds = patterns.read()

parameters = {
    "patternIds": patternIds
}

current_logs = requests.get(vehicleStatusesURL, params = parameters)

current_logs_json = requests.get(vehicleStatusesURL, params = parameters).json()

current_logs_json

[{'patternId': 11202,
  'name': '17',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMedium': False,
  'isFull': False,
  'headsignText': '7A - Trent U Gzowksi',
  'vehicleId': 5,
  'lastUpdate': '2026-10-09T21:25:32',
  'lat': 44.2669793,
  'lng': -78.3753345,
  'velocity': 10,
  'bearing': 4,
  'vehicleStateId': 1,
  'bypassDailyTripId': None},
 {'patternId': 11185,
  'name': '22',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMedium': False,
  'isFull': False,
  'headsignText': '6 - Fleming College',
  'vehicleId': 8,
  'lastUpdate': '2026-10-09T21:28:52',
  'lat': 44.3170813,
  'lng': -78.3199925,
  'velocity': 18,
  'bearing': 208,
  'vehicleStateId': 1,
  'bypassDailyTripId': None},
 {'patternId': 11190,
  'name': '45',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMed

---
Now that I know what the data looks like, it is time to create a system for fetching and organizing it.

I first want to manipulate the returned string and create a dictionary containing:

```
{
    "pattern_id",
    "vehicle_id",
    "route_id",
    "bearing",
    "headsign_text",
    "velocity",
    "lat",
    "lng",
    "last_update"
}
```

This will serve my facts table. My dimension table will be served by the reponse from api/Route, and change infrequently.

In [125]:
current_logs_json[0]

{'patternId': 11202,
 'name': '17',
 'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
 'vehicleCapacityIndicator': '',
 'isLight': False,
 'isMedium': False,
 'isFull': False,
 'headsignText': '7A - Trent U Gzowksi',
 'vehicleId': 5,
 'lastUpdate': '2026-10-09T21:25:32',
 'lat': 44.2669793,
 'lng': -78.3753345,
 'velocity': 10,
 'bearing': 4,
 'vehicleStateId': 1,
 'bypassDailyTripId': None}

In [126]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

PTBO_tz = ZoneInfo("America/Toronto")


# print(len(current_logs_json))

def get_current_logs():

    vehicle_logs = []

    server_time = requests.get("https://pt.mytransitride.com/api/SystemSettings").json()["currentServerTime"]

    print(f"Server Time: {server_time}")

    current_logs_json = requests.get(vehicleStatusesURL, params = parameters).json()

    for i in range(len(current_logs_json)):
        last_update = datetime.fromisoformat(current_logs_json[i]["lastUpdate"]).replace(tzinfo=timezone.utc)
        last_update = last_update.astimezone(PTBO_tz)
        vehicle_logs.append(
            {
                "pattern_id": current_logs_json[i]["patternId"],
                "vehicle_id": current_logs_json[i]["vehicleId"],
                "headsign_text": current_logs_json[i]["headsignText"],
                "velocity": current_logs_json[i]["velocity"],
                "bearing": current_logs_json[i]["bearing"],
                "lat": current_logs_json[i]["lat"],
                "lng": current_logs_json[i]["lng"],
                "last_update": last_update.isoformat(),
                "time_polled": server_time
            }
        )

    return vehicle_logs

print(get_current_logs()[0])

Server Time: 2026-10-09T17:29:02
{'pattern_id': 11202, 'vehicle_id': 5, 'headsign_text': '7A - Trent U Gzowksi', 'velocity': 10, 'bearing': 4, 'lat': 44.2669793, 'lng': -78.3753345, 'last_update': '2026-10-09T17:25:32-04:00', 'time_polled': '2026-10-09T17:29:02'}


---
Next, I'm going to write the logs to a file.

In [127]:
# with open(f"data/logs_write{datetime.now().strftime("%Y-%m-%dT%H-%M-%S")}.txt", "a+") as f:
#     for item in get_current_logs():
#         f.writelines(f"{item}\n")


# # # alternately:
# from datetime import date
import json

# def convert_logs_to_str(logs):
#     converted_logs = str(logs).replace("\'", "\"")
#     return converted_logs

with open(f"data/full_logs.json", "a+") as f:
    log = get_current_logs()
    # log = convert_logs_to_str(log)
    json.dump(log, f, indent = 2)

Server Time: 2026-10-09T17:29:02


The problem with the current approach is that:
1. I am creating a new file for every data fetch which will complicate loading the data into DB
2. From 1 above, I don't have a way to only append data which changed from the last fetch

I could:
- Load both the log file and the incoming data as dataframes and categorizing on `vehicle_id`, check if the new data's `last_update` is the same or different from the latest data's.

In [128]:
import pandas as pd

In [ ]:
# Something like:

def save_new_data():
    # get_current_logs()

    with open("data/incoming_data.json", "w+") as f:
        # incoming_data = convert_logs_to_str(get_current_logs())
        json.dump(get_current_logs(), f, indent = 2)

    new_log = pd.read_json("data/incoming_data.json")

    log = pd.read_json(f"data/full_logs.json")

    new_log["last_update"] = pd.to_datetime(new_log["last_update"], utc=True)
    new_log["time_polled"] = pd.to_datetime(new_log["time_polled"], utc=True)
    log["last_update"] = pd.to_datetime(log["last_update"], utc=True)
    log["time_polled"] = pd.to_datetime(log["time_polled"], utc=True)



    log["vehicle_id"] = log["vehicle_id"].astype("category")
    log["pattern_id"] = log["pattern_id"].astype("category")
    
    for vehicle_id in new_log["vehicle_id"].values:
        print(f"This is the examined v_id from incoming: {vehicle_id}\n")

        row_to_add = new_log.loc[new_log["vehicle_id"] == vehicle_id]
        print(f"This is the full row:\n {row_to_add}\nEnd of output\n\n")

        # if this is a new bus reporting data:
        if vehicle_id not in log["vehicle_id"].values:
            print("THIS SEEMS TO BE A NEW V_ID\n")
            print(f"These are all the v_ids in log:\n{log["vehicle_id"].values}\n End of output.\n\n")
            # append its data
            log = pd.concat([log, row_to_add], ignore_index=True)
        # elif this bus has already been reporting
        # if the data is new
        else:
            print("THIS SEEMS TO BE AN EXISTING V_ID\n")
            vehicle_most_recent = log.query("vehicle_id == @vehicle_id").nlargest(1, "last_update")
            print(f"This is the most recent v_id of same type in log: {vehicle_most_recent}\n")

            # vehicle in list, but differnt update tiems
            if row_to_add["last_update"].iloc[0] > vehicle_most_recent["last_update"].iloc[0]:
                log = pd.concat([log, row_to_add], ignore_index=True)
                print("WE ARE UPDATING LOG BECAUSE NEW DATA HAS MORE RECENT UPDATE TIME\n")
            # if update time is same, ignore
            else:
                print("WE ARE NOT UPDATING LOG.\n")
    
    log.to_json("data/full_logs.json", orient="records", indent = 2, date_format = "iso", index = False)
    
    print("Log File Updated")


In [136]:
save_new_data()

Server Time: 2026-10-09T17:34:44
This is the examined v_id from incoming: 5

This is the full row:
    pattern_id  vehicle_id         headsign_text  velocity  bearing        lat  \
0       11202           5  7A - Trent U Gzowksi         4       71  44.275624   

         lng               last_update               time_polled  
0 -78.371228 2026-10-09 21:34:33+00:00 2026-10-09 17:34:44+00:00  
End of output


THIS SEEMS TO BE AN EXISTING V_ID

This is the most recent v_id of same type in log:     pattern_id vehicle_id         headsign_text  velocity  bearing        lat  \
165      11202          5  7A - Trent U Gzowksi        20      203  44.267705   

           lng                      last_update  \
165 -78.376549 1970-01-01 00:29:51.581449+00:00   

                         time_polled  
165 1970-01-01 00:29:51.567074+00:00  

WE ARE UPDATING LOG BECAUSE NEW DATA HAS MORE RECENT UPDATE TIME

This is the examined v_id from incoming: 6

This is the full row:
    pattern_id  vehicle_i

C:\Users\djmck\AppData\Local\Temp\ipykernel_42412\2521929674.py:51: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  log.to_json("data/full_logs.json", orient="records", indent = 2, index = False)
